In [56]:
import torch.nn as nn
import pandas as pd
import json
import numpy as np

In [57]:
import torch
import torch.nn as nn


class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 1,
        hidden_dim: int = 64,
        latent_dim: int = 32,
        num_layers: int = 1,
    ):
        super().__init__()

        # ---------- Encoder ---------- #
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        # ---------- Bottleneck ---------- #
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)

        # ---------- Decoder ---------- #
        self.decoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(f"Expected (B, T, F), got {x.shape}")

        _, (h, _) = self.encoder(x)

        h = h[-1]  # (B, hidden)

        latent = self.to_latent(h)
        hidden_dec = self.from_latent(latent)

        seq_len = x.size(1)

        dec_in = torch.zeros_like(x)

        h0 = hidden_dec.unsqueeze(0).repeat(self.decoder.num_layers, 1, 1)
        c0 = torch.zeros_like(h0)

        out, _ = self.decoder(dec_in, (h0, c0))

        return out

In [58]:
import pandas as pd
import json
import numpy as np


# ---------- Load time series ---------- #
def load_series(csv_path: str):
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df


# ---------- Load anomaly windows ---------- #
def load_windows(json_path: str, file_key: str):
    with open(json_path, "r") as f:
        data = json.load(f)
    return data[file_key]


# ---------- Build point labels ---------- #
def build_labels(df, windows):
    labels = np.zeros(len(df))

    for start, end in windows:
        start = pd.to_datetime(start)
        end = pd.to_datetime(end)

        mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
        labels[mask] = 1

    df["label"] = labels
    return df


# ---------- Windowing ---------- #
def make_windows(values, labels, seq_len: int):
    x_windows = []
    y_windows = []

    for i in range(len(values) - seq_len + 1):
        x_windows.append(values[i:i + seq_len])
        y_windows.append(labels[i:i + seq_len].max())

    x_windows = np.stack(x_windows).astype(np.float32)
    y_windows = np.array(y_windows).astype(np.float32)

    return x_windows, y_windows

In [59]:
import torch
from torch.utils.data import DataLoader, TensorDataset


# ---------- Training loop ---------- #
def train_model(model, train_x, lr=1e-3, epochs=10, batch_size=64):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.MSELoss()

    model.train()

    x = torch.tensor(train_x, dtype=torch.float32)

    # ---------- Safety ---------- #
    if x.ndim == 2:
        x = x.unsqueeze(-1)

    if x.ndim != 3:
        raise ValueError(f"Invalid input shape: {x.shape}")

    dataset = TensorDataset(x)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0

        for (batch,) in loader:

            if batch.ndim == 2:
                batch = batch.unsqueeze(-1)

            pred = model(batch)
            loss = loss_fn(pred, batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        print(f"epoch {epoch+1}/{epochs} | loss {total_loss / max(n_batches,1):.6f}")

    return model


In [60]:
# ---------- Run ---------- #
SEQ_LEN = 30

df = load_series("../assets/data/art_daily_flatmiddle.csv")

windows = load_windows(
    json_path="../assets/labels/combined_windows.json",
    file_key="artificialWithAnomaly/art_daily_flatmiddle.csv"
)

df = build_labels(df, windows)

values = df["value"].values
labels = df["label"].values

x, y = make_windows(values, labels, SEQ_LEN)

# ---------- Train mask ---------- #
train_x = x[y == 0]

model = LSTMAutoencoder()
model = train_model(model, train_x, epochs=20)

c:\Users\omarg\.conda\envs\ts_lstm_ae\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([64, 30, 1])) that is different to the input size (torch.Size([64, 30, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
c:\Users\omarg\.conda\envs\ts_lstm_ae\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([51, 30, 1])) that is different to the input size (torch.Size([51, 30, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


epoch 1/20 | loss 2426.792622
epoch 2/20 | loss 2392.499699
epoch 3/20 | loss 2388.149582
epoch 4/20 | loss 2389.134883
epoch 5/20 | loss 2381.304674
epoch 6/20 | loss 2378.614785
epoch 7/20 | loss 2380.728348
epoch 8/20 | loss 2380.796467
epoch 9/20 | loss 2378.869727
epoch 10/20 | loss 2377.103995
epoch 11/20 | loss 2379.308498
epoch 12/20 | loss 2376.622687
epoch 13/20 | loss 2375.742264
epoch 14/20 | loss 2374.558690
epoch 15/20 | loss 2374.575684
epoch 16/20 | loss 2378.756960
epoch 17/20 | loss 2376.471970
epoch 18/20 | loss 2375.296749
epoch 19/20 | loss 2379.290057
epoch 20/20 | loss 2375.454008
